# **MICrONS project** : *Data analysis*
*katia russo, 2024-06-18*

---

Il progetto MICrONS (Machine Intelligence from Cortical Networks) include dataset di dati funzionali e strutturali provenienti da un campione di tessuto cerebrale di topo. Il dataset include immagini ad alta risoluzione della microstruttura del cervello, nonché registrazioni dell'attività neuronale. L'obiettivo principale è stato quello di analizzare questi dati per comprendere meglio la connettività e la funzionalità delle reti neurali nel cervello.

- Il mondo **funzionale**: neuroni vivi del topo vengono osservati con 2-photon calcium imaging mentre il topo guarda stimoli visivi, quindi filmati, pattern, clip. In pratica per ogni neurone ottieni una risposta nel tempo durante la presentazione degli stimoli. Nelle slide del corso è scritto esplicitamente che si tratta di topi GCaMP6s, quindi la parte funzionale riguarda neuroni eccitatori, con più sessioni e scansioni.
- Il mondo **strutturale/anatomico**: quella stessa regione di corteccia viene ricostruita con microscopia elettronica, segmentazione 3D, sinapsi, assoni, nuclei, ecc. Da lì arrivano informazioni come cell type anatomico, layer, area cerebrale, sinapsi, proofreading. Le slide spiegano anche che questo dataset cambia nel tempo: escono nuove versioni con segmentazione migliore, più proofreading e classificazioni più aggiornate.

Il problema grosso è che questi due mondi non nascono già perfettamente uniti. Per questo esiste il **matching** o coregistration: bisogna capire quale neurone osservato funzionalmente corrisponde a quale neurone ricostruito anatomicamente. (è preferibile usare quello manuale).


> Segmentazione: nella parte anatomica MICrONS si parte da un enorme volume di immagini di microscopia elettronica. La segmentazione è il processo con cui un algoritmo cerca di dire: “questi voxel/pixel appartengono allo stesso oggetto biologico”, quindi allo stesso neurone, assone, dendrite, glia, ecc.

> Proofreading: è la correzione manuale o semi-manuale di questi errori. Quindi dopo la segmentazione automatica, gli annotatori controllano e ripuliscono i risultati: correggono merge sbagliati, split sbagliati, ricostruzioni incomplete. Quando trovi scritto che un neurone o un assone è “proofread”, vuol dire che non ti stai fidando solo dell’algoritmo automatico, ma che c’è stato anche un controllo umano più accurato.


In [41]:
import microns_datacleaner as mic
import numpy as np
import pandas as pd

In [42]:
print("microns_datacleaner imported correctly")
print("MicronsDataCleaner class:", mic.MicronsDataCleaner)

microns_datacleaner imported correctly
MicronsDataCleaner class: <class 'microns_datacleaner.mic_datacleaner.MicronsDataCleaner'>


In [56]:
cleaner_min = mic.MicronsDataCleaner(datadir="data/data_min", version=1718, download_policy="minimum")
cleaner_all = mic.MicronsDataCleaner(datadir="data/data_all", version=1718, download_policy="all")

---

### **Tables exploration**

For a fixed version, there is a universe of available annotation tables.
- `get_table_list()` shows that universe.
- `download_policy`chooses a subset of that universe to download locally:
    - With `minimum`, the package downloads only the tables needed to build the unit table (deafulat option).
    - With `all`, it downloads all tables
    - With `extra`, it downloads the minimum set plus the specific tables you name in `extra_tables[]`.

In [ ]:
tables_1718 = cleaner_min.get_table_list() #it is indifferent if i used cleaner_all
n = len(tables_1718)
print(f"There are {n} tables available:\n{tables_1718}")

There are 45 tables available:
['synapses_pni_2', 'nucleus_detection_v0', 'vortex_manual_nodes_of_ranvier', 'bodor_pt_target_proofread', 'baylor_gnn_cell_type_fine_model_v2', 'nucleus_alternative_points', 'nucleus_functional_area_assignment', 'coregistration_auto_phase3_fwd_apl_vess_combined_v2', 'aibs_metamodel_mtypes_v661_v2_corrections', 'vortex_thalamic_proofreading_status', 'allen_column_mtypes_v2', 'proofreading_status_and_strategy', 'bodor_pt_cells', 'aibs_metamodel_mtypes_v661_v2', 'aibs_metamodel_celltypes_v661_corrections', 'vortex_microglia_proofreading_status', 'allen_v1_column_types_slanted_ref', 'multi_input_spine_predictions_ssa', 'aibs_column_nonneuronal_ref', 'nucleus_ref_neuron_svm', 'synapse_target_structure', 'myelin_auto_tags_2points', 'apl_functional_coreg_vess_fwd', 'vortex_axon_backtrace_column', 'cell_type_multifeature_combo', 'vortex_compartment_targets', 'baylor_log_reg_cell_type_coarse_v1', 'vortex_synapse_reattachment', 'coregistration_auto_phase3_fwd_v2', 

In [57]:
m = len(cleaner_min.tables_2_download)
print(f"Among all the tables, {m} are the one that `download_policy`=minimum select:\n{cleaner_min.tables_2_download}\n")

a = len(cleaner_all.tables_2_download)
print(f"Among all the tables, {a} are the one that `download_policy`=all select:\n{cleaner_all.tables_2_download}")

Among all the tables, 7 are the one that `download_policy`=minimum select:
['nucleus_detection_v0', 'proofreading_status_and_strategy', 'nucleus_functional_area_assignment', 'aibs_metamodel_celltypes_v661', 'digital_twin_properties_bcm_coreg_v4', 'coregistration_manual_v4', 'aibs_metamodel_celltypes_v661_corrections']

Among all the tables, 45 are the one that `download_policy`=all select:
['synapses_pni_2', 'nucleus_detection_v0', 'vortex_manual_nodes_of_ranvier', 'bodor_pt_target_proofread', 'baylor_gnn_cell_type_fine_model_v2', 'nucleus_alternative_points', 'nucleus_functional_area_assignment', 'coregistration_auto_phase3_fwd_apl_vess_combined_v2', 'aibs_metamodel_mtypes_v661_v2_corrections', 'vortex_thalamic_proofreading_status', 'allen_column_mtypes_v2', 'proofreading_status_and_strategy', 'bodor_pt_cells', 'aibs_metamodel_mtypes_v661_v2', 'aibs_metamodel_celltypes_v661_corrections', 'vortex_microglia_proofreading_status', 'allen_v1_column_types_slanted_ref', 'multi_input_spine_pr

- Per concreatamente downloadare le tabelle, bisogna utilizzare la funzione `download_tables([list of table_names])`, che permette di scaricare specifiche tabella.  
- La funzione `download_nucleus_data()` downloada tutte le nucleus-related tables, in base alla configurazione iniziale (-> sono le stesse tabelle che vengono scaricate se si sceglie `download_policy=minimum`).
- Per scaricare invece le tabelle relative alle sinapsi, bisogna usare `download_synapse_data(preids, postids)` che scarica solo le connections tra i neuroni specificati poiché l'intera tabella è enorme.

#### Nucleus-related tables:

In [50]:
cleaner_min.download_nucleus_data()

In [77]:
min_tables = cleaner_min.tables_2_download

for t in min_tables:
    df = cleaner_min.read_table(t)
    print(f"\n=== {t} ===")
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    display(df.head())


=== nucleus_detection_v0 ===
shape: (144120, 16)
columns: ['id', 'created', 'superceded_id', 'valid', 'volume', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'bb_start_position_x', 'bb_start_position_y', 'bb_start_position_z', 'bb_end_position_x', 'bb_end_position_y', 'bb_end_position_z', 'pt_supervoxel_id', 'pt_root_id']


,id,created,superceded_id,valid,volume,pt_position_x,pt_position_y,pt_position_z,bb_start_position_x,bb_start_position_y,bb_start_position_z,bb_end_position_x,bb_end_position_y,bb_end_position_z,pt_supervoxel_id,pt_root_id
0,730537,2020-09-28 22:40:41.780734+00:00,NaN,True,32.307938,381312,273984,19993,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,373879,2020-09-28 22:40:41.781788+00:00,NaN,True,229.045040,228816,239776,19593,NaN,NaN,NaN,NaN,NaN,NaN,96218056992431305,864691136090135607
2,601340,2020-09-28 22:40:41.782714+00:00,NaN,True,426.138000,340000,279152,20946,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,201858,2020-09-28 22:40:41.783784+00:00,NaN,True,93.753840,146848,213600,26267,NaN,NaN,NaN,NaN,NaN,NaN,84955554103121097,864691135373893678
4,600774,2020-09-28 22:40:41.785273+00:00,NaN,True,135.189790,339120,276112,19442,NaN,NaN,NaN,NaN,NaN,NaN,0,0



=== proofreading_status_and_strategy ===
shape: (2316, 14)
columns: ['id', 'created', 'superceded_id', 'valid', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'valid_id', 'status_dendrite', 'status_axon', 'strategy_dendrite', 'strategy_axon', 'pt_supervoxel_id', 'pt_root_id']


,id,created,superceded_id,valid,pt_position_x,pt_position_y,pt_position_z,valid_id,status_dendrite,status_axon,strategy_dendrite,strategy_axon,pt_supervoxel_id,pt_root_id
0,4584,2025-11-17 21:56:20.328504+00:00,NaN,True,296464,111200,16770,864691135686494647,True,True,dendrite_extended,axon_fully_extended,105489482232831453,864691135686494647
1,9,2024-06-03 19:45:52.508002+00:00,NaN,True,332369,118815,17518,864691136812081779,True,True,dendrite_extended,axon_interareal,110486693995521385,864691136812081779
2,14,2024-06-03 19:45:52.512832+00:00,NaN,True,173184,217472,21929,864691136195284556,True,True,dendrite_extended,axon_fully_extended,88615277952475942,864691136195284556
3,18,2024-06-03 19:45:52.516196+00:00,NaN,True,177437,213778,19873,864691135479404742,True,True,dendrite_extended,axon_interareal,89177746601127042,864691135479404742
4,19,2024-06-03 19:45:52.516972+00:00,NaN,True,332563,120074,18680,864691135975539779,True,True,dendrite_extended,axon_interareal,110486900288339126,864691135975539779



=== nucleus_functional_area_assignment ===
shape: (144120, 21)
columns: ['id_ref', 'created_ref', 'valid_ref', 'volume', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'bb_start_position_x', 'bb_start_position_y', 'bb_start_position_z', 'bb_end_position_x', 'bb_end_position_y', 'bb_end_position_z', 'pt_supervoxel_id', 'pt_root_id', 'id', 'created', 'tag', 'valid', 'target_id', 'value']


...

,id_ref,created_ref,valid_ref,volume,pt_position_x,pt_position_y,pt_position_z,bb_start_position_x,bb_start_position_y,bb_start_position_z,...,pref_dir,gOSI,gDSI,cc_abs,cc_max,cc_norm,OSI,DSI,readout_loc_x,readout_loc_y
0,335649,2020-09-28 22:41:20.303372+00:00,True,295.86110,210784,182032,22673,NaN,NaN,NaN,...,336.51070,0.099423,0.093671,0.500382,0.754597,0.663112,0.189591,0.248792,-0.144415,0.059087
1,194144,2020-09-28 22:42:01.511773+00:00,True,213.30724,136400,170640,17951,NaN,NaN,NaN,...,304.72702,0.351441,0.054476,0.432421,0.534646,0.808799,0.610376,0.134852,-0.245233,-0.201346
2,224395,2020-09-28 22:41:32.572651+00:00,True,329.44833,149840,133152,22592,NaN,NaN,NaN,...,303.05573,0.323088,0.106403,0.153038,0.299115,0.511636,0.556770,0.271835,-0.214071,-0.052871
3,488652,2020-09-28 22:41:44.769373+00:00,True,328.24426,287536,142464,19382,NaN,NaN,NaN,...,182.98495,0.182729,0.237040,0.330611,0.639865,0.516689,0.363019,0.598630,-0.026693,0.027254
4,332833,2020-09-28 22:44:41.864456+00:00,True,274.41873,209328,174304,20004,NaN,NaN,NaN,...,266.79608,0.379714,0.195747,0.363243,0.551817,0.658267,0.637795,0.463199,-0.151180,-0.083915



=== coregistration_manual_v4 ===
shape: (19181, 25)
columns: ['id', 'created', 'valid', 'target_id', 'session', 'scan_idx', 'unit_id', 'field', 'residual', 'score', 'id_ref', 'created_ref', 'valid_ref', 'volume', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'bb_start_position_x', 'bb_start_position_y', 'bb_start_position_z', 'bb_end_position_x', 'bb_end_position_y', 'bb_end_position_z', 'pt_supervoxel_id', 'pt_root_id']


,id,created,valid,target_id,session,scan_idx,unit_id,field,residual,score,...,pt_position_y,pt_position_z,bb_start_position_x,bb_start_position_y,bb_start_position_z,bb_end_position_x,bb_end_position_y,bb_end_position_z,pt_supervoxel_id,pt_root_id
0,5491,2024-05-21 18:38:25.372047+00:00,True,335649,6,2,6883,6,7.41244,2.60806,...,182032,22673,NaN,NaN,NaN,NaN,NaN,NaN,93747454767483710,864691135702330235
1,12542,2024-05-21 18:42:40.285576+00:00,True,194144,7,4,9575,6,8.55708,-0.71490,...,170640,17951,NaN,NaN,NaN,NaN,NaN,NaN,83542405709639148,864691135614842827
2,15097,2024-05-21 18:42:41.703496+00:00,True,194144,8,5,8632,6,4.25055,7.87525,...,170640,17951,NaN,NaN,NaN,NaN,NaN,NaN,83542405709639148,864691135614842827
3,12829,2024-05-21 18:42:40.443453+00:00,True,517966,7,5,1526,2,5.82370,4.16608,...,115888,16752,NaN,NaN,NaN,NaN,NaN,NaN,107530794289274882,864691136966116814
4,10490,2024-05-21 18:42:39.170710+00:00,True,224395,7,3,2398,2,7.02217,-1.39408,...,133152,22592,NaN,NaN,NaN,NaN,NaN,NaN,85366977140498420,864691135686521271



=== aibs_metamodel_celltypes_v661_corrections ===
shape: (347, 21)
columns: ['id_ref', 'created_ref', 'valid_ref', 'volume', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'bb_start_position_x', 'bb_start_position_y', 'bb_start_position_z', 'bb_end_position_x', 'bb_end_position_y', 'bb_end_position_z', 'pt_supervoxel_id', 'pt_root_id', 'id', 'created', 'valid', 'target_id', 'classification_system', 'cell_type']


,id_ref,created_ref,valid_ref,volume,pt_position_x,pt_position_y,pt_position_z,bb_start_position_x,bb_start_position_y,bb_start_position_z,...,bb_end_position_y,bb_end_position_z,pt_supervoxel_id,pt_root_id,id,created,valid,target_id,classification_system,cell_type
0,399356,2020-09-28 22:41:12.527331+00:00,True,221.04868,229872,186368,23181,NaN,NaN,NaN,...,NaN,NaN,96351648124987952,864691136137334155,1,2024-02-01 19:24:57.054004+00:00,True,399356,nonneuron,OPC
1,70708,2020-09-28 22:40:46.662047+00:00,True,177.97841,96368,159600,18950,NaN,NaN,NaN,...,NaN,NaN,78052200688939393,864691136043452246,2,2024-02-01 19:24:57.054558+00:00,True,70708,nonneuron,OPC
2,565608,2020-09-28 22:42:46.597055+00:00,True,67.90169,311552,238112,18365,NaN,NaN,NaN,...,NaN,NaN,107617587189819663,864691135717889684,3,2024-02-01 19:24:57.055086+00:00,True,565608,nonneuron,microglia
3,302387,2020-09-28 22:42:58.818118+00:00,True,80.29962,181808,194880,17267,NaN,NaN,NaN,...,NaN,NaN,89808522342513116,864691135526444123,4,2024-02-01 19:24:57.055641+00:00,True,302387,nonneuron,microglia
4,135499,2020-09-28 22:40:58.686762+00:00,True,106.97491,114352,266016,15532,NaN,NaN,NaN,...,NaN,NaN,80529399916752128,864691135699652642,5,2024-02-01 19:24:57.056163+00:00,True,135499,nonneuron,oligo


Le 7 tabelle minime non sono casuali. Insieme costruiscono esattamente questa pipeline logica:
1. `nucleus_detection_v0n`: elenco base dei nuclei
2. `proofreading_status_and_strategy`: qualità/revisione della ricostruzione
3. `nucleus_functional_area_assignment`: area funzionale (eg: V1, LM, AL, ecc.)
4. `aibs_metamodel_celltypes_v661`: label di cell_type (eg. 23P, 5P-IT) e classification_system (eg. exitatory)
5. `aibs_metamodel_celltypes_v661_corrections`: correzioni delle label
6. `coregistration_manual_v4`: matching manuale con functional. “questo nucleo anatomico corrisponde a questa unità funzionale in questa sessione/scan”.
7. `digital_twin_properties_bcm_coreg_v4`: feature funzionali derivate sui neuroni coregistrati  

Questa è praticamente la logica interna di costruzione di `units`che vedremo dopo.

In [79]:
df_nucleus = cleaner_min.read_table("nucleus_detection_v0")
df_proof = cleaner_min.read_table("proofreading_status_and_strategy")
df_area = cleaner_min.read_table("nucleus_functional_area_assignment")
df_celltypes = cleaner_min.read_table("aibs_metamodel_celltypes_v661")
df_dt = cleaner_min.read_table("digital_twin_properties_bcm_coreg_v4")
df_coreg = cleaner_min.read_table("coregistration_manual_v4")
df_corr = cleaner_min.read_table("aibs_metamodel_celltypes_v661_corrections")

In [83]:
print("Nuclei totali:", len(df_nucleus))
print("Unique pt_root_id:", df_nucleus['pt_root_id'].nunique())

print("\nFunctional areas:")
print(df_area['tag'].value_counts())

print("\nCelltypes:")
print("Unique nuclei:", df_celltypes['target_id'].nunique())
print(df_celltypes['classification_system'].value_counts())
print(df_celltypes['cell_type'].value_counts())

print("\nCoregistration manual:")
print("Rows:", len(df_coreg))
print("Unique nuclei:", df_coreg['target_id'].nunique())

print("\nDigital twin:")
print("Rows:", len(df_dt))
print("Unique nuclei:", df_dt['target_id'].nunique())

Nuclei totali: 144120
Unique pt_root_id: 121403

Functional areas:
tag
V1    90765
RL    31622
AL    20859
LM      874
Name: count, dtype: int64

Celltypes:
Unique nuclei: 94010
classification_system
excitatory_neuron    64195
nonneuron            21856
inhibitory_neuron     7963
Name: count, dtype: int64
cell_type
23P          19735
4P           14777
6P-IT        11734
5P-IT         7949
astrocyte     7850
oligo         7020
6P-CT         6815
BC            3365
pericyte      2645
microglia     2638
MC            2466
5P-ET         2215
OPC           1703
BPC           1488
5P-NP          970
NGC            644
Name: count, dtype: int64

Coregistration manual:
Rows: 19181
Unique nuclei: 15439

Digital twin:
Rows: 15780
Unique nuclei: 13090


#### *Extra tables*:

In [ ]:
#i've excluded the synapses tables from this exploration because they are too large, along with some other large table
extra_useful_tables = [
    'cell_type_multifeature_combo',
    'cg_cell_type_calls',
    'baylor_log_reg_cell_type_coarse_v1',
    'baylor_gnn_cell_type_fine_model_v2',
    'aibs_metamodel_mtypes_v661_v2',
    'aibs_metamodel_mtypes_v661_v2_corrections',
    'coregistration_auto_phase3_fwd_v2',
    'coregistration_auto_phase3_fwd_apl_vess_combined_v2',
    'apl_functional_coreg_vess_fwd',
    'gamlin_2023_mcs',
    'gamlin_2023_mcs_met_types',
    'allen_column_mtypes_v2',
    'allen_v1_column_types_slanted_ref'
]

cleaner_all.download_tables(extra_useful_tables)

In [78]:
for t in extra_useful_tables:
    df = cleaner_all.read_table(t)
    print(f"\n=== {t} ===")
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    display(df.head())


=== cell_type_multifeature_combo ===
shape: (69712, 21)
columns: ['id', 'created', 'valid', 'target_id', 'classification_system', 'cell_type', 'id_ref', 'created_ref', 'valid_ref', 'volume', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'bb_start_position_x', 'bb_start_position_y', 'bb_start_position_z', 'bb_end_position_x', 'bb_end_position_y', 'bb_end_position_z', 'pt_supervoxel_id', 'pt_root_id']


,id,created,valid,target_id,classification_system,cell_type,id_ref,created_ref,valid_ref,volume,...,pt_position_y,pt_position_z,bb_start_position_x,bb_start_position_y,bb_start_position_z,bb_end_position_x,bb_end_position_y,bb_end_position_z,pt_supervoxel_id,pt_root_id
0,1,2026-02-26 06:27:05.803317+00:00,True,291161,excitatory,L2IT,291161,2020-09-28 22:41:20.559577+00:00,True,322.96353,...,108896,21755,NaN,NaN,NaN,NaN,NaN,NaN,90148821795140164,864691136010651052
1,2,2026-02-26 06:27:05.803998+00:00,True,261392,excitatory,L4IT,261392,2020-09-28 22:41:30.176102+00:00,True,298.27660,...,157872,27288,NaN,NaN,NaN,NaN,NaN,NaN,89240557209214977,864691136287385283
2,3,2026-02-26 06:27:05.804574+00:00,True,583521,excitatory,L3IT,583521,2020-09-28 22:44:45.961693+00:00,True,281.17056,...,138688,17577,NaN,NaN,NaN,NaN,NaN,NaN,110630111543546271,864691135446112658
3,4,2026-02-26 06:27:05.805134+00:00,True,424705,excitatory,L4IT,424705,2020-09-28 22:44:57.621420+00:00,True,301.51804,...,140496,26808,NaN,NaN,NaN,NaN,NaN,NaN,99160282328485886,864691135695975834
4,5,2026-02-26 06:27:05.805672+00:00,True,619424,excitatory,L6IT,619424,2020-09-28 22:44:32.919193+00:00,True,262.26523,...,204272,17378,NaN,NaN,NaN,NaN,NaN,NaN,113453657336912736,864691135104768077



=== cg_cell_type_calls ===
shape: (2868, 23)
columns: ['id', 'created', 'valid', 'target_id', 'classification_system', 'cell_type', 'id_ref', 'created_ref', 'valid_ref', 'pre_pt_position_x', 'pre_pt_position_y', 'pre_pt_position_z', 'post_pt_position_x', 'post_pt_position_y', 'post_pt_position_z', 'ctr_pt_position_x', 'ctr_pt_position_y', 'ctr_pt_position_z', 'size', 'pre_pt_supervoxel_id', 'pre_pt_root_id', 'post_pt_supervoxel_id', 'post_pt_root_id']


,id,created,valid,target_id,classification_system,cell_type,id_ref,created_ref,valid_ref,pre_pt_position_x,...,post_pt_position_y,post_pt_position_z,ctr_pt_position_x,ctr_pt_position_y,ctr_pt_position_z,size,pre_pt_supervoxel_id,pre_pt_root_id,post_pt_supervoxel_id,post_pt_root_id
0,1,2022-02-07 19:04:03+00:00,True,252841327,cg_calls,4P,252841327,2020-11-04 14:33:23.840014+00:00,True,230014,...,177168,22727,230072,177245,22728,1264.0,96420779851496531,864691135361291591,96420779851511006,864691135570035334
1,2,2022-02-07 19:04:03+00:00,True,252889242,cg_calls,5P_IT,252889242,2020-11-04 10:33:52.615201+00:00,True,229016,...,183316,23710,228982,183326,23713,896.0,96280867131039747,864691135361291591,96280867131044154,864691135778536365
2,3,2022-02-07 19:04:03+00:00,True,234601405,cg_calls,23P,234601405,2020-11-04 11:41:10.073096+00:00,True,222018,...,89748,22713,221960,89754,22717,1576.0,95283128914105798,864691135361291591,95283128914108860,864691135408808649
3,4,2022-02-07 19:04:03+00:00,True,248722118,cg_calls,INH,248722118,2020-11-04 13:28:14.430823+00:00,True,229408,...,89958,22389,229462,89936,22389,5028.0,96268359985173049,864691135361291591,96338728729319889,864691135064994884
4,5,2022-02-07 19:04:03+00:00,True,220312874,cg_calls,INH,220312874,2020-11-04 08:51:40.223504+00:00,True,215122,...,87900,21903,215156,87862,21907,4356.0,94368128947379613,864691135361291591,94368128947360899,864691135776899245



=== baylor_log_reg_cell_type_coarse_v1 ===
shape: (55063, 21)
columns: ['id', 'created', 'valid', 'target_id', 'classification_system', 'cell_type', 'id_ref', 'created_ref', 'valid_ref', 'volume', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'bb_start_position_x', 'bb_start_position_y', 'bb_start_position_z', 'bb_end_position_x', 'bb_end_position_y', 'bb_end_position_z', 'pt_supervoxel_id', 'pt_root_id']


...

,id,created,tag,valid,target_id,value,id_ref,created_ref,valid_ref,volume,...,pt_position_y,pt_position_z,bb_start_position_x,bb_start_position_y,bb_start_position_z,bb_end_position_x,bb_end_position_y,bb_end_position_z,pt_supervoxel_id,pt_root_id
0,1,2024-12-18 00:36:09.413004+00:00,Sst-MET-4,True,398518,1.000,398518,2020-09-28 22:44:32.461253+00:00,True,260.78480,...,187856,19274,NaN,NaN,NaN,NaN,NaN,NaN,96562959979272019,864691134989909114
1,2,2024-12-18 00:36:09.413681+00:00,Sst-MET-4,True,400905,0.790,400905,2020-09-28 22:44:09.840387+00:00,True,223.67125,...,196896,19088,NaN,NaN,NaN,NaN,NaN,NaN,97478990603962239,864691135118298333
2,3,2024-12-18 00:36:09.414265+00:00,Sst-MET-9,True,404260,0.878,404260,2020-09-28 22:45:02.944995+00:00,True,312.13602,...,209360,24584,NaN,NaN,NaN,NaN,NaN,NaN,98043590562832435,864691135341516741
3,4,2024-12-18 00:36:09.414851+00:00,Sst-MET-6,True,269585,0.982,269585,2020-09-28 22:44:17.071361+00:00,True,236.81647,...,209344,22995,NaN,NaN,NaN,NaN,NaN,NaN,88403072342632329,864691135404765166
4,5,2024-12-18 00:36:09.415419+00:00,Sst-MET-8,True,161736,1.000,161736,2020-09-28 22:42:23.206468+00:00,True,260.95108,...,183248,18971,NaN,NaN,NaN,NaN,NaN,NaN,82558961412258145,864691135375430985



=== allen_column_mtypes_v2 ===
shape: (1351, 21)
columns: ['id_ref', 'created_ref', 'valid_ref', 'volume', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'bb_start_position_x', 'bb_start_position_y', 'bb_start_position_z', 'bb_end_position_x', 'bb_end_position_y', 'bb_end_position_z', 'pt_supervoxel_id', 'pt_root_id', 'id', 'created', 'valid', 'target_id', 'classification_system', 'cell_type']


,id_ref,created_ref,valid_ref,volume,pt_position_x,pt_position_y,pt_position_z,bb_start_position_x,bb_start_position_y,bb_start_position_z,...,bb_end_position_y,bb_end_position_z,pt_supervoxel_id,pt_root_id,id,created,valid,target_id,classification_system,cell_type
0,258319,2020-09-28 22:40:42.476911+00:00,True,261.80615,178400,143248,21238,NaN,NaN,NaN,...,NaN,NaN,89309001002848425,864691136968429774,99,2023-08-21 22:53:34.255741+00:00,True,258319,excitatory,L2c
1,276438,2020-09-28 22:40:42.700226+00:00,True,277.31772,179648,258768,23597,NaN,NaN,NaN,...,NaN,NaN,89465269428261699,864691135164434989,1160,2023-08-21 22:53:35.075622+00:00,True,276438,excitatory,L6tall-b
2,260552,2020-09-28 22:40:42.745779+00:00,True,230.11180,177408,157968,21002,NaN,NaN,NaN,...,NaN,NaN,89170256379033022,864691135778954848,100,2023-08-21 22:53:34.256631+00:00,True,260552,excitatory,L4a
3,260263,2020-09-28 22:40:42.746658+00:00,True,274.32420,169440,158128,20266,NaN,NaN,NaN,...,NaN,NaN,88044356338331571,864691135694415551,101,2023-08-21 22:53:34.257543+00:00,True,260263,excitatory,L3b
4,262898,2020-09-28 22:40:42.749245+00:00,True,230.09232,172512,175280,21964,NaN,NaN,NaN,...,NaN,NaN,88468836747612860,864691135778773856,1405,2023-08-21 22:53:35.262828+00:00,True,262898,inhibitory,ITC



=== allen_v1_column_types_slanted_ref ===
shape: (1357, 21)
columns: ['id_ref', 'created_ref', 'valid_ref', 'volume', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'bb_start_position_x', 'bb_start_position_y', 'bb_start_position_z', 'bb_end_position_x', 'bb_end_position_y', 'bb_end_position_z', 'pt_supervoxel_id', 'pt_root_id', 'id', 'created', 'valid', 'target_id', 'classification_system', 'cell_type']


,id_ref,created_ref,valid_ref,volume,pt_position_x,pt_position_y,pt_position_z,bb_start_position_x,bb_start_position_y,bb_start_position_z,...,bb_end_position_y,bb_end_position_z,pt_supervoxel_id,pt_root_id,id,created,valid,target_id,classification_system,cell_type
0,258319,2020-09-28 22:40:42.476911+00:00,True,261.80615,178400,143248,21238,NaN,NaN,NaN,...,NaN,NaN,89309001002848425,864691136968429774,50,2023-03-18 14:13:21.613360+00:00,True,258319,aibs_coarse_excitatory,23P
1,276438,2020-09-28 22:40:42.700226+00:00,True,277.31772,179648,258768,23597,NaN,NaN,NaN,...,NaN,NaN,89465269428261699,864691135164434989,1119,2023-03-18 14:13:22.506660+00:00,True,276438,aibs_coarse_excitatory,6P-CT
2,260552,2020-09-28 22:40:42.745779+00:00,True,230.11180,177408,157968,21002,NaN,NaN,NaN,...,NaN,NaN,89170256379033022,864691135778954848,35,2023-03-18 14:13:21.602813+00:00,True,260552,aibs_coarse_excitatory,23P
3,260263,2020-09-28 22:40:42.746658+00:00,True,274.32420,169440,158128,20266,NaN,NaN,NaN,...,NaN,NaN,88044356338331571,864691135694415551,95,2023-03-18 14:13:21.644304+00:00,True,260263,aibs_coarse_excitatory,23P
4,262898,2020-09-28 22:40:42.749245+00:00,True,230.09232,172512,175280,21964,NaN,NaN,NaN,...,NaN,NaN,88468836747612860,864691135778773856,81,2023-03-18 14:13:21.634505+00:00,True,262898,aibs_coarse_inhibitory,BPC


In [33]:
print(cleaner.info_to_correct)

{'aibs_metamodel_celltypes_v661_corrections': ['classification_system', 'cell_type']}


Now let's inspect all the 

In [14]:
units, segments = cleaner.process_nucleus_data(functional_data=None)

print(units.shape)
print(units.columns.tolist())

Transform positions: 100%|██████████| 94014/94014 [00:00<00:00, 135332.49it/s]


(90434, 12)
['nucleus_id', 'pt_root_id', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'classification_system', 'cell_type', 'brain_area', 'strategy_axon', 'strategy_dendrite', 'tuning_type', 'layer']
